In [5]:
import pandas as pd
import numpy as np 

class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

def calculate_gini(y):
    class_counts = {}
    for label in y:
        if label in class_counts:
            class_counts[label] += 1
        else:
            class_counts[label] = 1
    total_samples = len(y)
    sum_of_squared_probabilities = 0.0
    
    for label in class_counts:
        probability = class_counts[label] / total_samples
        sum_of_squared_probabilities += (probability * probability)        
    gini = 1.0 - sum_of_squared_probabilities
    return gini

def get_best_split(X, y):
    best_feature = None
    best_threshold = None
    lowest_gini = float('inf')
    
    total_rows = len(X)
    total_columns = len(X[0])
    for feature_index in range(total_columns):
        column_values = X[:, feature_index]
        unique_thresholds = np.unique(column_values)
        for threshold in unique_thresholds:
            left_y = []
            right_y = []            
            for row_index in range(total_rows):
                if column_values[row_index] <= threshold:
                    left_y.append(y[row_index])
                else:
                    right_y.append(y[row_index])
            if len(left_y) == 0 or len(right_y) == 0:
                continue
            gini_left = calculate_gini(left_y)
            gini_right = calculate_gini(right_y)
            weight_left = len(left_y) / total_rows
            weight_right = len(right_y) / total_rows
            weighted_gini = (weight_left * gini_left) + (weight_right * gini_right)
            if weighted_gini < lowest_gini:
                lowest_gini = weighted_gini
                best_feature = feature_index
                best_threshold = threshold
    return best_feature, best_threshold

def build_tree(X, y, depth=0, max_depth=5, min_samples_split=2):
    total_samples = len(y)    
    all_same_class = True
    first_label = y[0]
    for label in y:
        if label != first_label:
            all_same_class = False
            break            
    if depth >= max_depth or total_samples < min_samples_split or all_same_class:
        class_counts = {}
        for label in y:
            if label in class_counts:
                class_counts[label] += 1
            else:
                class_counts[label] = 1
        most_common = max(class_counts, key=class_counts.get)
        return Node(value=most_common)        
    feature_index, threshold = get_best_split(X, y)    
    if feature_index is None:
        class_counts = {}
        for label in y:
            class_counts[label] = class_counts.get(label, 0) + 1
        most_common = max(class_counts, key=class_counts.get)
        return Node(value=most_common)        
    left_X, left_y = [], []
    right_X, right_y = [], []
    for row_index in range(total_samples):
        if X[row_index, feature_index] <= threshold:
            left_X.append(X[row_index])
            left_y.append(y[row_index])
        else:
            right_X.append(X[row_index])
            right_y.append(y[row_index])            
    left_X = np.array(left_X)
    left_y = np.array(left_y)
    right_X = np.array(right_X)
    right_y = np.array(right_y)    
    left_branch = build_tree(left_X, left_y, depth + 1, max_depth, min_samples_split)
    right_branch = build_tree(right_X, right_y, depth + 1, max_depth, min_samples_split)
    return Node(feature_index=feature_index, threshold=threshold, left=left_branch, right=right_branch)

def predict_single_row(node, row_data):
    current_node = node    
    while current_node.value is None:
        feature_value = row_data[current_node.feature_index]
        if feature_value <= current_node.threshold:
            current_node = current_node.left
        else:
            current_node = current_node.right
    return current_node.value
def predict(tree, X_test):
    predictions = []
    for row in X_test:
        pred = predict_single_row(tree, row)
        predictions.append(pred)
    return np.array(predictions)

def calculate_macro_f1(actual_labels, predicted_labels):
    unique_classes = []
    for label in actual_labels:
        if label not in unique_classes:
            unique_classes.append(label)
    all_f1_scores = []
    for target_class in unique_classes:
        true_positives = 0
        false_positives = 0
        false_negatives = 0        
        for i in range(len(actual_labels)):
            actual = actual_labels[i]
            prediction = predicted_labels[i]
            if actual == target_class and prediction == target_class:
                true_positives += 1
            elif actual != target_class and prediction == target_class:
                false_positives += 1
            elif actual == target_class and prediction != target_class:
                false_negatives += 1
        if true_positives + false_positives > 0:
            precision = true_positives / (true_positives + false_positives)
        else:
            precision = 0.0
        if true_positives + false_negatives > 0:
            recall = true_positives / (true_positives + false_negatives)
        else:
            recall = 0.0            
        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0.0
        all_f1_scores.append(f1)        
    total = sum(all_f1_scores)/ len(all_f1_scores)
    return total 

def k_fold_cross_validation(X, y, k=5, max_depth=5):
    total_rows = len(X)
    fold_size = total_rows // k
    all_fold_scores = []
    for fold_number in range(k):
        start_index = fold_number * fold_size        
        if fold_number == k - 1:
            end_index = total_rows
        else:
            end_index = start_index + fold_size            
        X_val, y_val = [], []
        for i in range(start_index, end_index):
            X_val.append(X[i])
            y_val.append(y[i])            
        X_train, y_train = [], []
        for i in range(total_rows):
            if i < start_index or i >= end_index:
                X_train.append(X[i])
                y_train.append(y[i])                
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        X_val = np.array(X_val)
        y_val = np.array(y_val)        
        my_tree = build_tree(X_train, y_train, max_depth=max_depth)        
        predictions = predict(my_tree, X_val)        
        fold_score = calculate_macro_f1(y_val, predictions)
        all_fold_scores.append(fold_score)
        print(f"Fold {fold_number + 1} Score: {fold_score:.4f}")        
    average_score = sum(all_fold_scores) / len(all_fold_scores)
    print(f"\nFinal Cross-Validation F1 Score: {average_score:.4f}")
    
    return average_score
df = pd.read_csv('yeast.csv')
X = df[['mcg', 'gvh', 'alm', 'mit', 'erl', 'pox', 'vac', 'nuc']].values
y = df['name'].values
final_score = k_fold_cross_validation(X, y, k=5, max_depth=5)
%pwd

Fold 1 Score: 0.4927
Fold 2 Score: 0.3982
Fold 3 Score: 0.3727
Fold 4 Score: 0.4172
Fold 5 Score: 0.4673

Final Cross-Validation F1 Score: 0.4296


'/home/raghav-m/envs/jupyter/Task2.1'